# PhenoAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.PhenoAge)

class PhenoAge(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        """Apply Levine's natural-log transform to C-reactive protein.

        The published coefficient is fit against ln(CRP in mg/dL); users supply
        the raw measurement so the same column can feed clocks that log it
        differently.

        Notes
        -----
        CRP is clamped to ``CRP_FLOOR_MG_DL`` before the log, so a zero — from a
        below-detection reading, a constant imputer, or a column the input never
        had — yields a finite age instead of ``-inf`` propagating into every
        downstream summary. Clamping is safe precisely because the floor equals
        the registered lower bound: any value it moves is already out of range
        and has already been warned about.
        """
        index = self.features.index("c_reactive_protein")
        crp = torch.clamp(x[:, index : index + 1], min=CRP_FLOOR_MG_DL)
        return torch.cat([x[:

In [3]:
model = pya.models.PhenoAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "phenoage"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: blood chemistry
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2018
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Levine, M. E., et al. \"An epigenetic biomarker of aging for lifespan and healthspan.\" Aging 10.4 (2018): 573-591."
model.metadata["doi"] = "https://doi.org/10.18632/aging.101414"
model.metadata["notes"] = "Clinical Phenotypic Age combines chronological age with nine blood biomarkers selected by penalized mortality regression and expresses mortality risk as an equivalent age in years. C-reactive protein is supplied raw in mg/dL and natural-log-transformed inside the clock, with readings at or below 0.01 mg/dL clamped to that floor; before pyaging 0.5.0 the feature was named log_crp and the caller had to supply the log. Continuing to pass a pre-logged value biases the estimate downward: any pre-logged CRP below 1.01 mg/dL, which covers most healthy adults, lands on the clamp floor and returns the same answer as omitting CRP entirely — about four years younger than the correct value for a typical adult, so the estimate reads healthier than it is."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood"]  # Paper: blood
model.metadata["predicts"] = ["phenotypic age"]  # Paper: phenotypic age
model.metadata["training_target"] = ["mortality"]  # Paper: all-cause mortality
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "penalized hazards regression with Gompertz calibration"  # Paper: Penalized proportional hazards regression with Gompertz calibration
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: clinical laboratory assays
model.metadata["population"] = "adults"  # Paper: NHANES III adults aged 20 years or older (n=9,926)
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 10
model.metadata["citations"] = 3594
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
features = [
    "albumin",
    "creatinine",
    "glucose",
    "c_reactive_protein",
    "lymphocyte_percent",
    "mean_cell_volume",
    "red_cell_distribution_width",
    "alkaline_phosphatase",
    "white_blood_cell_count",
    "age"
]

coefs = [
    -0.0336,
    0.0095,
    0.1953,
    0.0954,
    -0.0120,
    0.0268,
    0.3306,
    0.0019,
    0.0554,
    0.0804,
]

#### A note on C-reactive protein units

## Load features

In [6]:
model.features = features

## Load weights into base model

In [7]:
weights = torch.tensor(coefs).unsqueeze(0)
intercept = torch.tensor([-19.9067])

#### Linear model

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "natural_log_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'mortality_to_phenoage'
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Levine, M. E., et al. "An epigenetic biomarker of aging for '
             'lifespan and healthspan." Aging 10.4 (2018): 573-591.',
 'citations': 3594,
 'citations_date': '2026-07-05',
 'clock_name': 'phenoage',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.18632/aging.101414',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'penalized hazards regression with Gompertz calibration',
 'n_features': 10,
 'notes': 'Clinical Phenotypic Age combines chronological age with nine blood '
          'biomarkers selected by penalized mortality regression and expresses '
          'mortality risk as an equivalent age in years. C-reactive protein is '
          'supplied raw in mg/dL and natural-log-transformed inside the clock, '
          'with readings at or below 0.01 mg/dL clamped 

## Normal feature ranges

In [13]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

,feature,unit,low,high
0,albumin,g/L,10.00,70.0
1,creatinine,umol/L,10.00,3000.0
2,glucose,mmol/L,1.00,60.0
3,c_reactive_protein,mg/dL,0.01,50.0
4,lymphocyte_percent,%,0.00,100.0


## Basic test

In [14]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

tensor([[inf],
        [inf],
        [inf],
        [inf],
        [inf],
        [inf],
        [inf],
        [inf],
        [inf],
        [inf]], dtype=torch.float64, grad_fn=<AddBackward0>)

## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)